In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

def load_block_as_pyg(base_dir, block, feature_cols=None):
    df_x = pd.read_csv(os.path.join(base_dir, block, "features", "net_features_place.csv"))
    df_y = pd.read_csv(os.path.join(base_dir, block, "route_label", "net_labels.csv"))
    df_e = pd.read_csv(os.path.join(base_dir, block, "graph", "edge_index.csv"))

    df_x["net_name"] = df_x["net_name"].astype(str).str.strip()
    df_y["net_name"] = df_y["net_name"].astype(str).str.strip()
    df_e["src_net"]  = df_e["src_net"].astype(str).str.strip()
    df_e["dst_net"]  = df_e["dst_net"].astype(str).str.strip()

    df = df_x.merge(df_y, on="net_name", how="inner")
    df = df.drop_duplicates("net_name").reset_index(drop=True)

    # Choose numeric feature columns
    non_feat = {"net_name","net_dbid","odb_net","status","label"}
    if feature_cols is None:
        cols = [c for c in df.columns if c not in non_feat]
    else:
        cols = feature_cols

    X = df[cols].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(0.0).astype(np.float32)

    y = df["label"].astype(int).values

    # net_name -> node index
    name2idx = {n:i for i,n in enumerate(df["net_name"].tolist())}

    # edges: keep only edges where both ends exist in df
    src = []
    dst = []
    for s,t in zip(df_e["src_net"], df_e["dst_net"]):
        if s in name2idx and t in name2idx:
            src.append(name2idx[s])
            dst.append(name2idx[t])

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    x = torch.tensor(X.values, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    data.net_name = df["net_name"].tolist()
    return data, cols


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class NetSAGE(nn.Module):
    def __init__(self, in_dim, hidden=64, layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden))
        for _ in range(layers-2):
            self.convs.append(SAGEConv(hidden, hidden))
        self.convs.append(SAGEConv(hidden, hidden))
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        logits = self.head(x).squeeze(-1)
        return logits


In [4]:
def compute_pos_weight(y):
    y = y.cpu().numpy()
    pos = y.sum()
    neg = len(y) - pos
    return torch.tensor([neg / max(1, pos)], dtype=torch.float32)

In [17]:
import os
import torch
import numpy as np
from collections import defaultdict

#designs="\
#usb_phy \
#pcm_slv_top \
#FIR_filter \
#spi_top \
#simple_spi_top \
#sasc_top \
#i2c_master_top \
#ac97_top \
#mc_top \
#sha256 \
#"

BASE = "../dataset"

train_blocks = [
    "pcm_slv_top", 'usb_phy', 'spi_top', 'simple_spi_top', 'i2c_master_top', 'sasc_top', 'mc_top', 'sha256',
]

test_blocks = [
    "FIR_filter", 'ac97_top',
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
from torch_geometric.data import Data

def load_block(base, block, feature_cols=None):
    import pandas as pd

    df_x = pd.read_csv(f"{base}/{block}/features/net_features_place.csv")
    df_y = pd.read_csv(f"{base}/{block}/route_label/net_labels.csv")
    df_e = pd.read_csv(f"{base}/{block}/graph/edge_index.csv")

    df = df_x.merge(df_y, on="net_name", how="inner")
    df = df.drop_duplicates("net_name").reset_index(drop=True)

    non_feat = {"net_name","net_dbid","odb_net","status","label"}
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c not in non_feat]

    X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    y = df["label"].values.astype(np.int64)

    name2idx = {n:i for i,n in enumerate(df["net_name"])}

    src, dst = [], []
    for s,t in zip(df_e["src_net"], df_e["dst_net"]):
        if s in name2idx and t in name2idx:
            src.append(name2idx[s])
            dst.append(name2idx[t])

    data = Data(
        x = torch.tensor(X.values, dtype=torch.float32),
        edge_index = torch.tensor([src, dst], dtype=torch.long),
        y = torch.tensor(y, dtype=torch.long)
    )
    data.block = block
    data.net_names = df["net_name"].tolist()

    return data, feature_cols

In [19]:
train_graphs = []
feature_cols = None

for b in train_blocks:
    g, feature_cols = load_block(BASE, b, feature_cols)
    train_graphs.append(g)

test_graphs = []
for b in test_blocks:
    g, _ = load_block(BASE, b, feature_cols)
    test_graphs.append(g)

print("Features:", len(feature_cols))

Features: 66


In [20]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class NetSAGE(nn.Module):
    def __init__(self, in_dim, hidden=64, layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden))
        for _ in range(layers-1):
            self.convs.append(SAGEConv(hidden, hidden))

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x).squeeze(-1)

In [21]:
def compute_pos_weight(graphs):
    pos, neg = 0, 0
    for g in graphs:
        pos += int(g.y.sum())
        neg += int((g.y == 0).sum())
    return torch.tensor([neg / max(1, pos)], dtype=torch.float32)

pos_weight = compute_pos_weight(train_graphs).to(device)
print("pos_weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

pos_weight: 24.47077178955078


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NetSAGE(
    in_dim=len(feature_cols),
    hidden=64,
    layers=3,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

In [23]:
def train_epoch(graphs):
    model.train()
    total_loss = 0.0

    for g in graphs:
        g = g.to(device)
        optimizer.zero_grad()
        logits = model(g.x, g.edge_index)
        loss = criterion(logits, g.y.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(graphs)

In [24]:
for epoch in range(1, 51):
    loss = train_epoch(train_graphs)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:03d} | loss = {loss:.4f}")

Epoch 005 | loss = 1.2576
Epoch 010 | loss = 1.1861
Epoch 015 | loss = 1.2100
Epoch 020 | loss = 1.1849
Epoch 025 | loss = 1.1720
Epoch 030 | loss = 1.1824
Epoch 035 | loss = 1.1748
Epoch 040 | loss = 1.1725
Epoch 045 | loss = 1.2516
Epoch 050 | loss = 1.1781


In [25]:
from sklearn.metrics import average_precision_score, roc_auc_score

def eval_graph(g):
    model.eval()
    g = g.to(device)
    with torch.no_grad():
        logits = model(g.x, g.edge_index)
        probs = torch.sigmoid(logits).cpu().numpy()
        y = g.y.cpu().numpy()

    pr = average_precision_score(y, probs)
    roc = roc_auc_score(y, probs)

    K = int(y.sum())
    idx = np.argsort(-probs)[:K]
    recall_k = y[idx].sum() / max(1, K)

    return pr, roc, recall_k, K

print("\nTEST RESULTS")
for g in test_graphs:
    pr, roc, rK, K = eval_graph(g)
    print(f"{g.block:12s} | PR-AUC={pr:.3f} ROC-AUC={roc:.3f} Recall@{K}={rK:.3f}")


TEST RESULTS
FIR_filter   | PR-AUC=0.076 ROC-AUC=0.511 Recall@291=0.058
ac97_top     | PR-AUC=0.110 ROC-AUC=0.495 Recall@838=0.237


In [26]:
print(model)

NetSAGE(
  (convs): ModuleList(
    (0): SAGEConv(66, 64, aggr=mean)
    (1-2): 2 x SAGEConv(64, 64, aggr=mean)
  )
  (head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
